# Module 17 — LoRA & QLoRA: Parameter-Efficient Fine-Tuning

**Part IV · Scaling & Efficiency · 25–35 min**

---

You have a 70B model. You want to customize it on your company's tickets, or your legal corpus, or that weird internal DSL nobody outside your org has ever seen.

You do **NOT** have 140 GB of VRAM lying around. You do NOT want to pay $40k for 8× H100s just to burn a weekend on a fine-tune that might not even work. You probably have one 24 GB consumer card, or a rented A100 at best.

**What do?**

Option A: shard the optimizer state across nodes, cry about activation checkpointing, pay the cloud bill, wait three days. Full fine-tuning of a 70B model needs roughly **12× the model size** in memory (weights + grads + Adam moments + activations). That's ~1.4 TB. Nope.

Option B: notice that the *update* you're applying — `ΔW` — is much simpler than the full weight matrix. It turns out `ΔW` is essentially **low rank**. You don't need 4096×4096 degrees of freedom to teach the model your ticket format. You need maybe 16. Maybe 64 on a hard day.

That's LoRA. And when you combine it with 4-bit quantization of the frozen base, that's QLoRA — the technique that single-handedly made fine-tuning 65B models on a single 24 GB GPU a *weekend project*.

This notebook:

1. Builds the math of low-rank updates from first principles
2. Shows *why* fine-tuning updates are low-rank (empirically — we'll measure it)
3. Fine-tunes GPT-2 three ways: full, LoRA r=4/16/64, and compares memory + quality
4. Visualizes a LoRA update as a **thin lens** bolted onto the frozen matrix
5. Merges LoRA back into the base — zero inference overhead
6. Covers the practical decisions: which layers, what rank, what alpha, when DoRA helps


## 0 · Setup

We use plain PyTorch plus `transformers` for GPT-2. Everything here runs on CPU in a few minutes; a GPU just makes it snappier.

In [ ]:
import math, time, copy, os, gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
print("torch:", torch.__version__)

## 1 · The core insight: fine-tuning updates are low rank

When you fine-tune a pretrained model, you start from $W_0$ and end at $W_0 + \Delta W$. The pretrained weights already encode "how language works". The update $\Delta W$ only has to encode the *delta* — "here's how THIS customer's tickets differ from generic English".

That delta is vastly simpler than $W_0$ itself. Aghajanyan et al. (2020) showed pretrained models have low **intrinsic dimension**: you can reparameterize the whole fine-tune through a random projection of a few thousand parameters and get most of the performance back. Hu et al. (2021) — the LoRA paper — made this practical.

### The LoRA reparameterization

Instead of learning the full matrix $\Delta W \in \mathbb{R}^{d \times d}$, factor it:

$$\Delta W = B A, \qquad B \in \mathbb{R}^{d \times r},\ A \in \mathbb{R}^{r \times d},\ r \ll d$$

The forward pass becomes:

$$h = W_0 x + \Delta W x = W_0 x + B A x$$

with $W_0$ **frozen**. Only $A$ and $B$ get gradients.

### Why this is a huge win

For a $d \times d$ matrix, full fine-tuning has $d^2$ trainable params. LoRA has $2 d r$. At $d = 4096$ and $r = 16$:

$$\frac{2 \cdot 4096 \cdot 16}{4096^2} = \frac{2 r}{d} = \frac{32}{4096} \approx 0.78\%$$

And that's *per matrix*. Across the whole model (we don't even touch embeddings or MLPs usually), LoRA typically trains **0.1–1% of total parameters**.

### Initialization matters

We init $A$ from a small Gaussian and $B = 0$. That means at step 0, $BA = 0$ exactly — the model is unchanged. Training has to *earn* every bit of the update. If you init both randomly, you destroy the pretrained weights on the first forward pass.

### The $\alpha$ scaling trick

In practice we use $\Delta W = \frac{\alpha}{r} B A$. The $\alpha / r$ factor decouples the effective learning rate from the rank: doubling $r$ roughly halves the per-parameter gradient magnitude, so scaling by $\alpha/r$ keeps the update magnitude comparable across ranks. Standard practice: set $\alpha = r$ or $\alpha = 2r$ and forget about it.


## 2 · Warmup: how good is a rank-r approximation, actually?

Before we commit to low-rank updates, let's see empirically how well low-rank matrices can approximate things. We'll take a pretrained GPT-2 attention matrix and ask: *what fraction of its energy lives in the top-r singular values?*

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load GPT-2 small (124M). Frozen for now.
model_name = "gpt2"
tok = GPT2Tokenizer.from_pretrained(model_name)
tok.pad_token = tok.eos_token
base_model = GPT2LMHeadModel.from_pretrained(model_name)
base_model.eval()
n_params = sum(p.numel() for p in base_model.parameters())
print(f"loaded {model_name}: {n_params/1e6:.1f}M params")

In [ ]:
# Grab the attention projection from a middle layer.
layer = base_model.transformer.h[6]
# GPT-2 uses Conv1D — weight is stored transposed. We want W such that y = x @ W.
W_attn = layer.attn.c_attn.weight.detach().float().cpu().numpy()  # (768, 2304) = q,k,v concat
print("W_attn shape:", W_attn.shape)

U, S, Vt = np.linalg.svd(W_attn, full_matrices=False)
total_energy = (S**2).sum()
cum_energy = np.cumsum(S**2) / total_energy
print(f"top 16 singular values capture {cum_energy[15]*100:.1f}% of energy")
print(f"top 64 singular values capture {cum_energy[63]*100:.1f}% of energy")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].semilogy(S, lw=1.5)
axes[0].set_xlabel("singular value index")
axes[0].set_ylabel("magnitude (log)")
axes[0].set_title("GPT-2 layer 6 c_attn: singular value spectrum")
axes[0].grid(alpha=0.3)

axes[1].plot(cum_energy, lw=1.5)
for r, c in [(4, "C1"), (16, "C2"), (64, "C3")]:
    axes[1].axvline(r, color=c, ls="--", alpha=0.7, label=f"r={r}: {cum_energy[r-1]*100:.0f}%")
axes[1].set_xlabel("rank")
axes[1].set_ylabel("cumulative energy fraction")
axes[1].set_title("how much of the matrix lives in the top-r subspace")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Read this carefully.** A rank-16 approximation of the *full* attention matrix only captures a modest fraction — maybe 20–30% of the Frobenius energy. That's not great.

But here's the twist: **LoRA doesn't approximate $W$. LoRA approximates $\Delta W$.** The *delta*, not the weights themselves. The delta turns out to be *much* more low-rank than $W$, because fine-tuning really is a small correction to a model that already mostly works. We'll verify this experimentally in section 5.

## 3 · Building a LoRA layer from scratch

Enough talk. Here's the whole thing in 25 lines. We wrap an existing `nn.Linear`, freeze it, and add two small trainable matrices on the side.

In [ ]:
class LoRALinear(nn.Module):
    """Wraps an existing nn.Linear with a low-rank trainable update.

    Forward:  y = x @ W0^T + b  +  (alpha/r) * (x @ A^T) @ B^T
    Only A and B receive gradients. W0 is frozen.
    """
    def __init__(self, base_linear: nn.Linear, r: int = 8, alpha: int = 16, dropout: float = 0.0):
        super().__init__()
        self.base = base_linear
        for p in self.base.parameters():
            p.requires_grad = False

        in_f, out_f = base_linear.in_features, base_linear.out_features
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        # A: (r, in_f), kaiming init. B: (out_f, r), zero init.
        self.A = nn.Parameter(torch.empty(r, in_f))
        self.B = nn.Parameter(torch.zeros(out_f, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        out = self.base(x)
        lora = self.drop(x) @ self.A.T @ self.B.T
        return out + self.scaling * lora

    def merged_weight(self):
        """Return W0 + (alpha/r) * B @ A — for merging post-training."""
        return self.base.weight.data + self.scaling * (self.B @ self.A)

    def extra_repr(self):
        return f"r={self.r}, alpha={self.alpha}, trainable={self.A.numel()+self.B.numel()}"

# sanity check: a frozen base + zero-init B means the module starts as identity.
lin = nn.Linear(128, 256)
lora = LoRALinear(lin, r=4, alpha=8)
x = torch.randn(3, 128)
assert torch.allclose(lora(x), lin(x)), "LoRA should be a no-op at init"
print("LoRA at init equals base:", torch.allclose(lora(x), lin(x)))
print(lora)

Two details worth internalizing:

1. **B is zero at init.** The LoRA path contributes nothing initially. You never risk smashing the pretrained weights.
2. **Everything downstream of A and B flows through a rank-$r$ bottleneck.** No matter what the network does later, the LoRA update itself has rank at most $r$.

## 4 · LoRA-fy GPT-2

GPT-2 uses `Conv1D` (a transposed linear) rather than `nn.Linear`. We'll patch the attention QKV projection (`c_attn`) and the output projection (`c_proj`) of each block — the two layers that are most commonly targeted in practice.

Empirically, LoRA-ing the attention projections gives you most of the benefit. Adding MLPs helps a bit more. Adding embeddings usually doesn't.

In [ ]:
from transformers.pytorch_utils import Conv1D

class LoRAConv1D(nn.Module):
    """Conv1D stores weight as (in_f, out_f) and computes x @ W + b.

    We bolt a rank-r update onto it.
    """
    def __init__(self, base: Conv1D, r: int = 8, alpha: int = 16, dropout: float = 0.0):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        in_f = base.weight.shape[0]
        out_f = base.weight.shape[1]
        self.r, self.alpha, self.scaling = r, alpha, alpha / r
        self.A = nn.Parameter(torch.empty(r, in_f))
        self.B = nn.Parameter(torch.zeros(out_f, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        base_out = self.base(x)  # (..., out_f)
        lora_out = self.drop(x) @ self.A.T @ self.B.T
        return base_out + self.scaling * lora_out

    def merged_weight(self):
        # base.weight is (in_f, out_f). Our update is (A^T B^T) = A.T @ B.T, shape (in_f, out_f).
        return self.base.weight.data + self.scaling * (self.A.T @ self.B.T)

def apply_lora_to_gpt2(model, r=8, alpha=16, targets=("c_attn", "c_proj"), dropout=0.0):
    n_wrapped = 0
    for block in model.transformer.h:
        for name in targets:
            # c_proj exists on both attn and mlp — we target attention only by default
            if name == "c_attn":
                parent, attr = block.attn, "c_attn"
            elif name == "c_proj":
                parent, attr = block.attn, "c_proj"
            else:
                continue
            mod = getattr(parent, attr)
            if isinstance(mod, Conv1D):
                setattr(parent, attr, LoRAConv1D(mod, r=r, alpha=alpha, dropout=dropout))
                n_wrapped += 1
    # freeze everything else
    for n, p in model.named_parameters():
        if "A" not in n and "B" not in n:
            p.requires_grad = False
        # (LoRA A/B params have 'A' or 'B' in their fully-qualified name)
    # re-enable trainable on just A/B
    for m in model.modules():
        if isinstance(m, LoRAConv1D):
            m.A.requires_grad = True
            m.B.requires_grad = True
    return n_wrapped

def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
def count_total(model):
    return sum(p.numel() for p in model.parameters())

In [ ]:
# Quick demo: LoRA-fy a fresh copy, count params
demo = GPT2LMHeadModel.from_pretrained(model_name)
n_w = apply_lora_to_gpt2(demo, r=16, alpha=32)
trainable = count_trainable(demo)
total = count_total(demo)
print(f"wrapped {n_w} matrices")
print(f"trainable params: {trainable:,}  ({100*trainable/total:.3f}% of total {total:,})")
del demo; gc.collect()

That 0.1–0.5% number is the whole pitch. You can keep the full 124M model in memory, you're only optimizing a few hundred thousand parameters, so:

- **Adam state is tiny** (2 floats per trainable param instead of 2 × 124M)
- **Gradients are tiny** (same story)
- **The frozen base doesn't need grad bookkeeping at all**

## 5 · The experiment: fine-tune GPT-2 three ways

We'll fine-tune GPT-2 small on a tiny synthetic "dataset" — a handful of sentences in a made-up style the model has never seen. The goal isn't SOTA, it's to compare:

1. **Full fine-tune** — every parameter trainable
2. **LoRA r=4** — minimal adapter
3. **LoRA r=16** — the default
4. **LoRA r=64** — overkill

Across: trainable parameter count, peak memory, training loss, final generation quality, and wall-clock time.

In [ ]:
# A deliberately weird mini-corpus: pirate Shakespeare tech support.
corpus = [
    "Arr matey, hast thou tried turning it off and on again, i' faith?",
    "Verily, the kernel panic doth haunt this vessel like a ghost.",
    "Forsooth, thy RAM be swollen, and the ship's log overfloweth.",
    "Yo ho ho, a segfault on the starboard buffer, all hands to gdb!",
    "Mine liege, the CPU doth smoke, and the fans do wail a mournful shanty.",
    "Hark! A null pointer hath been spotted off the port bow!",
    "By yon cursed race condition, the deadlock hath claimed another thread.",
    "Shiver me timbers, the cache be cold and the disk be full of barnacles.",
    "Zounds, the stack hath overflowed and the heap lies in tatters.",
    "Thou scurvy dog of a linter, cease thy whining about mine indentation.",
    "Aye, the container doth float, but the pod refuseth to schedule.",
    "The merge conflict rageth like a tempest upon the main branch.",
    "Tis a bug most foul, hiding in yon regex like a kraken in the deep.",
    "Prithee restart the daemon, lest the port remain forever bound.",
    "Good sirrah, thy YAML indentation hath summoned a parser curse.",
    "Avast! The TLS handshake hath failed in the fifth fathom of the OSI stack.",
]

def make_batch(seq_len=64, batch_size=8):
    # Tokenize everything and chunk.
    text = " ".join(corpus * 20)  # repeat a bit
    ids = tok(text, return_tensors="pt").input_ids[0]
    # random crops
    out = []
    for _ in range(batch_size):
        start = torch.randint(0, max(1, len(ids) - seq_len - 1), (1,)).item()
        out.append(ids[start:start+seq_len+1])
    out = torch.stack(out)
    return out[:, :-1].to(device), out[:, 1:].to(device)

x, y = make_batch()
print("batch:", x.shape, y.shape)

In [ ]:
def train_run(model, tag, steps=80, lr=5e-4, batch_size=8, seq_len=64):
    model.to(device).train()
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(trainable_params, lr=lr)

    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()

    t0 = time.time()
    losses = []
    for step in range(steps):
        x, y = make_batch(seq_len=seq_len, batch_size=batch_size)
        out = model(x, labels=y)
        loss = out.loss
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    elapsed = time.time() - t0

    peak_mem = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else float("nan")
    trainable = sum(p.numel() for p in trainable_params)
    total = sum(p.numel() for p in model.parameters())

    print(f"[{tag:>14}] trainable={trainable:>10,} "
          f"({100*trainable/total:5.2f}%)  "
          f"final_loss={losses[-1]:.3f}  "
          f"wall={elapsed:5.1f}s  "
          f"peak_vram={peak_mem:.2f}GB")
    return dict(tag=tag, losses=losses, trainable=trainable, total=total,
                peak_mem=peak_mem, wall=elapsed, model=model)

In [ ]:
runs = []

# 1. Full fine-tune
full = GPT2LMHeadModel.from_pretrained(model_name)
for p in full.parameters(): p.requires_grad = True
runs.append(train_run(full, "full", steps=80, lr=5e-5))

# 2. LoRA r=4
m4 = GPT2LMHeadModel.from_pretrained(model_name)
apply_lora_to_gpt2(m4, r=4, alpha=8)
runs.append(train_run(m4, "lora r=4", steps=80, lr=5e-4))

# 3. LoRA r=16
m16 = GPT2LMHeadModel.from_pretrained(model_name)
apply_lora_to_gpt2(m16, r=16, alpha=32)
runs.append(train_run(m16, "lora r=16", steps=80, lr=5e-4))

# 4. LoRA r=64
m64 = GPT2LMHeadModel.from_pretrained(model_name)
apply_lora_to_gpt2(m64, r=64, alpha=128)
runs.append(train_run(m64, "lora r=64", steps=80, lr=5e-4))

A few things to notice in the printout:

- LoRA runs train **0.1–2%** of the params of the full run.
- Final loss for LoRA is *comparable* (sometimes better, sometimes slightly worse). On a tiny dataset and only 80 steps this is noisy — but the gap is tiny relative to the param reduction.
- LoRA wall-clock is usually *faster* per step because the optimizer step is trivial and there's less grad bookkeeping.
- If you ran this on GPU you'd see peak VRAM drop by a factor of 3–5× for LoRA.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Loss curves
for r in runs:
    axes[0].plot(r["losses"], label=r["tag"], lw=1.5)
axes[0].set_xlabel("step")
axes[0].set_ylabel("training loss")
axes[0].set_title("training loss: full vs LoRA ranks")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Trainable parameter counts (log)
tags = [r["tag"] for r in runs]
train_counts = [r["trainable"] for r in runs]
colors = ["C3", "C0", "C1", "C2"]
axes[1].bar(tags, train_counts, color=colors)
axes[1].set_yscale("log")
axes[1].set_ylabel("trainable params (log)")
axes[1].set_title("trainable parameter count")
for i, v in enumerate(train_counts):
    axes[1].text(i, v, f"{v/1e6:.2f}M" if v > 1e6 else f"{v/1e3:.0f}K",
                 ha="center", va="bottom", fontsize=9)

# Trainable as fraction of total
fracs = [100 * r["trainable"] / r["total"] for r in runs]
axes[2].bar(tags, fracs, color=colors)
axes[2].set_ylabel("% of total params")
axes[2].set_title("trainable fraction")
for i, v in enumerate(fracs):
    axes[2].text(i, v, f"{v:.2f}%", ha="center", va="bottom", fontsize=9)

for ax in axes[1:]:
    ax.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()

### Does any of this actually *work*? Quick qualitative check.

In [ ]:
def sample(model, prompt, max_new=30, temperature=0.8):
    model.eval()
    ids = tok(prompt, return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        out = model.generate(
            ids, max_new_tokens=max_new, do_sample=True,
            temperature=temperature, top_p=0.9, pad_token_id=tok.eos_token_id
        )
    return tok.decode(out[0], skip_special_tokens=True)

prompt = "Arr matey, the database"
torch.manual_seed(42)
for r in runs:
    print(f"\n[{r['tag']}]")
    print("  " + sample(r["model"], prompt))

All four variants should produce something mildly pirate-Shakespearean. The full fine-tune may sound a hair more fluent; the LoRA r=16 and r=64 should be indistinguishable in quality. The r=4 run may sometimes slip back into generic English — that's your signal that rank 4 is underpowered for this task.

## 6 · The "thin lens" view of a LoRA update

Here's the surprise moment. Pick one LoRA-fied matrix from our trained r=16 model. Materialize the update $\Delta W = (\alpha/r) B A$ and look at it next to the frozen base.

Think of the frozen $W_0$ as a thick, opaque lens already ground for "speak English". The LoRA update $\Delta W$ is a **thin corrective lens** — a plate of glass only 16 rays wide — bolted onto the front. All the heavy lifting happens in $W_0$; $\Delta W$ just bends the beam slightly toward "pirate Shakespeare".

In [ ]:
# Grab LoRA deltas from the r=16 run for layer 6.
m = runs[2]["model"]  # lora r=16
layer6 = m.transformer.h[6].attn.c_attn  # this is a LoRAConv1D now
assert isinstance(layer6, LoRAConv1D)

W0 = layer6.base.weight.detach().float().cpu().numpy()
A = layer6.A.detach().float().cpu().numpy()
B = layer6.B.detach().float().cpu().numpy()
dW = (layer6.scaling) * (A.T @ B.T)  # shape matches W0: (in_f, out_f)

print("W0 frobenius norm:", np.linalg.norm(W0))
print("ΔW frobenius norm:", np.linalg.norm(dW))
print("ratio ||ΔW|| / ||W0||:", np.linalg.norm(dW) / np.linalg.norm(W0))
print("ΔW matrix rank (numerical):", np.linalg.matrix_rank(dW, tol=1e-6))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

vmax = np.percentile(np.abs(W0), 99)
axes[0].imshow(W0[:128, :128], cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
axes[0].set_title("frozen base W0 (crop)")
axes[0].set_xlabel("out"); axes[0].set_ylabel("in")

vmax_d = np.percentile(np.abs(dW), 99) + 1e-8
axes[1].imshow(dW[:128, :128], cmap="RdBu_r", vmin=-vmax_d, vmax=vmax_d, aspect="auto")
axes[1].set_title(f"LoRA ΔW (crop)  ||ΔW||/||W0||={np.linalg.norm(dW)/np.linalg.norm(W0):.3f}")
axes[1].set_xlabel("out"); axes[1].set_ylabel("in")

# Merged
W_new = W0 + dW
axes[2].imshow(W_new[:128, :128], cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
axes[2].set_title("merged W0 + ΔW (crop)")
axes[2].set_xlabel("out"); axes[2].set_ylabel("in")

plt.tight_layout(); plt.show()

In [ ]:
# Singular value spectrum of ΔW vs W0 — the money plot.
U_d, S_d, _ = np.linalg.svd(dW, full_matrices=False)
U_b, S_b, _ = np.linalg.svd(W0, full_matrices=False)

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.semilogy(S_b / S_b[0], label="W0 (frozen base)", lw=1.5)
ax.semilogy(S_d / (S_d[0] + 1e-12), label="ΔW (LoRA update)", lw=1.8, color="C1")
ax.axvline(16, color="k", ls="--", alpha=0.5, label="r=16 cutoff")
ax.set_xlabel("singular value index")
ax.set_ylabel("normalized magnitude (log)")
ax.set_title("ΔW has exactly rank 16. W0 has full rank.")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

The $\Delta W$ spectrum has a hard cliff at rank 16 — by construction, it can't have more non-zero singular values. The $W_0$ spectrum keeps going all the way to 768.

That cliff is the thin lens. The pretrained weights form a dense, high-rank refraction of language; you bolted a 16-dim corrective plate on top and it was enough to teach the model to curse in Elizabethan English.

## 7 · Merging LoRA back: zero inference overhead

A LoRA adapter adds a branch to the forward pass. That's roughly a 10–20% slowdown at inference. You don't pay this in production: you compute

$$W_{\text{merged}} = W_0 + \frac{\alpha}{r} B A$$

once, copy it into the original `Conv1D`/`Linear`, and throw the LoRA modules away. The merged model is bit-for-bit a regular GPT-2 — any inference engine can serve it with zero adapter plumbing.

In [ ]:
def merge_lora(model):
    """Swap every LoRAConv1D back for a plain Conv1D with merged weights."""
    merged_count = 0
    for block in model.transformer.h:
        for attr in ("c_attn", "c_proj"):
            mod = getattr(block.attn, attr)
            if isinstance(mod, LoRAConv1D):
                new_w = mod.merged_weight()
                # Build a vanilla Conv1D with the same shape
                merged = Conv1D(mod.base.weight.shape[1], mod.base.weight.shape[0])
                merged.weight.data.copy_(new_w)
                merged.bias.data.copy_(mod.base.bias.data)
                setattr(block.attn, attr, merged)
                merged_count += 1
    return merged_count

# Take the trained r=16 model, make a deep copy, merge.
to_merge = copy.deepcopy(runs[2]["model"]).to(device).eval()
pre_out = None
test_ids = tok("Arr matey, the database", return_tensors="pt").input_ids.to(device)
with torch.no_grad():
    pre_logits = to_merge(test_ids).logits

n_merged = merge_lora(to_merge)
print(f"merged {n_merged} LoRA modules back into Conv1D")

with torch.no_grad():
    post_logits = to_merge(test_ids).logits

diff = (pre_logits - post_logits).abs().max().item()
print(f"max |pre - post| logit diff after merge: {diff:.2e}")
print("(should be tiny — floating-point noise only)")

That max-diff is essentially numerical noise. The merged model is functionally identical to the LoRA-branched one but has zero runtime overhead. Any inference stack that can serve GPT-2 can serve your fine-tune.

You can also keep the adapter *separate* and hot-swap it. Inference servers like vLLM and SGLang support "multi-LoRA" where one base model is shared across hundreds of tenants, each with their own tiny adapter that gets swapped in per request. That's how you run a SaaS where every customer has a bespoke model on top of one GPU.

## 8 · QLoRA: LoRA meets 4-bit quantization

LoRA already shrinks the *trainable* state. But the frozen base still has to sit somewhere, and for a 65B model "somewhere" is 130 GB in fp16. QLoRA (Dettmers et al. 2023) attacks this:

1. **Quantize the frozen base to 4-bit NormalFloat (NF4)** — ~0.5 GB per billion params, so 65B → ~33 GB
2. **Keep LoRA adapters in bf16** — they're tiny, precision matters for the update
3. **Page the optimizer state to CPU** when GPU memory gets tight ("paged optimizers")
4. **Double-quantize** the quantization constants themselves (saves another ~0.4 bits/param)

The surprising finding: NF4 is a near-information-theoretic optimal 4-bit code for the kind of weight distributions neural nets actually have (roughly Gaussian, zero-mean). It preserves quality better than int4 and even better than fp4 at the same bit count.

**Memory budget for QLoRA on 65B:**

| Component | fp16 | QLoRA |
|---|---|---|
| Frozen weights | 130 GB | 33 GB (NF4) |
| Gradients | 130 GB | ~0.2 GB (LoRA only) |
| Adam moments | 260 GB | ~0.4 GB (LoRA only) |
| Activations (ckpt) | ~8 GB | ~8 GB |
| **Total** | **~530 GB** | **~42 GB** |

And suddenly a 65B fine-tune fits on a **single 48 GB A6000**, or with a bit more care on a 24 GB 3090. This is not a marginal improvement — it's a different category of what's possible.

### Why we're not running QLoRA live

The full QLoRA stack depends on `bitsandbytes`, which wants a CUDA GPU and matching CUDA toolchain. That's a lot of machine-specific setup for a curriculum notebook. Below we *simulate* the memory math and the precision cost, which gives you the right mental model without the install pain.

In [ ]:
# Simulate NF4 quantization: the NF4 codebook from the QLoRA paper.
NF4_LEVELS = torch.tensor([
    -1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
    -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
    0.07958029955625534, 0.16093020141124725, 0.24611230194568634,
    0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
    0.7229568362236023, 1.0
])

def nf4_quantize(w, block_size=64):
    """Quantize w to NF4 codes blockwise. Returns dequantized w'."""
    flat = w.flatten()
    pad = (-len(flat)) % block_size
    if pad:
        flat = torch.cat([flat, torch.zeros(pad)])
    blocks = flat.view(-1, block_size)
    scales = blocks.abs().max(dim=1, keepdim=True).values.clamp(min=1e-8)
    normed = blocks / scales
    # Find nearest NF4 level for each element
    idx = (normed.unsqueeze(-1) - NF4_LEVELS).abs().argmin(dim=-1)
    deq = NF4_LEVELS[idx] * scales
    deq = deq.flatten()[:w.numel()].view_as(w)
    return deq

W_test = torch.from_numpy(W0).float()
W_nf4 = nf4_quantize(W_test)
rel_err = (W_test - W_nf4).norm() / W_test.norm()
print(f"NF4 relative reconstruction error on GPT-2 c_attn: {rel_err:.4f}")
print(f"original bits/param: 16 (fp16) or 32 (fp32)")
print(f"NF4 bits/param: 4 (+ ~0.5 for scales with double-quant)")

In [ ]:
# Memory comparison: model size vs trainable size, full vs LoRA vs QLoRA.
sizes = [(7, "7B"), (13, "13B"), (34, "34B"), (65, "65B"), (70, "70B")]
names = [s[1] for s in sizes]

full_fp16    = [s[0] * 12 for s in sizes]    # weights + grad + adam ≈ 12x params in fp16
lora_fp16    = [s[0] * 2 + 0.5 for s in sizes]  # frozen fp16 + tiny adapter overhead
qlora        = [s[0] * 0.55 + 0.5 for s in sizes]  # NF4 frozen + adapter

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(names))
w = 0.27
ax.bar(x - w, full_fp16, w, label="full fine-tune (fp16)", color="C3")
ax.bar(x,     lora_fp16, w, label="LoRA (fp16 base)", color="C0")
ax.bar(x + w, qlora,     w, label="QLoRA (NF4 base)", color="C2")
ax.axhline(24, color="k", ls="--", alpha=0.6, label="24 GB (consumer GPU)")
ax.axhline(80, color="gray", ls=":", alpha=0.6, label="80 GB (H100)")
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylabel("approximate peak VRAM (GB)")
ax.set_title("fine-tuning memory: full vs LoRA vs QLoRA")
ax.set_yscale("log")
ax.legend(loc="upper left", fontsize=9)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

The dashed line at 24 GB is where things get interesting. Without QLoRA, you top out at fine-tuning 7B on a single consumer GPU — and that's tight. With QLoRA you're comfortably fine-tuning 34B on the same card, and a 65B model is *almost* in reach with aggressive activation checkpointing and paging.

## 9 · Practical LoRA: what actually matters

After you've done this a few times, you converge on a small set of defaults.

### Which layers?

- **Always:** attention projections. `q_proj`, `k_proj`, `v_proj`, `o_proj` (or their GPT-2 `c_attn`/`c_proj` equivalents).
- **Usually:** the MLP projections too. Especially `down_proj`/`up_proj` if you want a bit more capacity. Costs a bit more memory but helps most tasks.
- **Rarely:** embeddings. They're huge, and you usually don't need to adjust them. Exception: adding a new language / tokenizer vocabulary.
- **Never:** layer norms. Too few params to bother, and they're already fine.

### Rank $r$

- **r=4–8**: classification, style transfer, simple format adaptation
- **r=16–32**: most instruction-tuning jobs, this is the default
- **r=64–128**: continued pre-training on a new domain (medical, legal, code)
- **r=256+**: you almost never need this. Consider DoRA or full fine-tuning instead.

### Alpha $\alpha$

Just set $\alpha = 2r$ and move on. The paper's sweep showed it barely matters as long as you scale it with $r$. If your loss diverges, halve $\alpha$; if it's not learning, double it.

### Learning rate

An order of magnitude higher than full fine-tuning. `1e-4` to `5e-4` is typical. LoRA is remarkably forgiving of high LR because the adapter is initialized at zero — you can't wreck the base.

### Dropout

`0.05` to `0.1` on the LoRA path acts as regularization on small datasets. Turn it off for large ones.

### Merge or not merge?

- **Merge** for single-tenant production: zero latency hit.
- **Don't merge** if you need multi-tenant hot-swapping or continual retraining. vLLM's multi-LoRA lets you serve 100+ adapters on one base.

### Gotchas

- **LoRA is dtype-sensitive.** Train the adapter in bf16, not fp16 — fp16 can underflow the small gradients.
- **Don't forget to save the base config** alongside your adapter. An adapter is meaningless without the exact base it was trained against.
- **Validation loss plateaus fast.** LoRA typically converges in 1–3 epochs on a few thousand examples. If you're training for longer, you probably want more data, not more steps.


## 10 · Beyond vanilla LoRA: a brief taxonomy

The LoRA family has grown. Quick mental map:

- **DoRA** (Weight-Decomposed LoRA, 2024). Decomposes $W = m \cdot \frac{V}{\|V\|}$ into a magnitude and direction, then applies LoRA to the direction only. Closes most of the remaining gap to full fine-tuning at the same rank. Default choice in 2026 for quality-sensitive jobs.
- **LoRA+**. Uses a *different* learning rate for $B$ vs $A$ — specifically a larger one for $B$. Turns out the optimal ratio is ~16×. Free ~1-2% quality bump, zero compute cost. Just do it.
- **AdaLoRA**. Allocates rank budget across layers *adaptively*: sensitive layers get more rank, boring ones get less. Prunes singular values during training. Useful when you're rank-budget-constrained and want the model to tell you where to spend.
- **VeRA** (Vector-based Random Matrix Adaptation, 2024). Shares random $A$ and $B$ matrices across all layers and only learns per-layer scaling vectors. 10× fewer trainable params than LoRA. Surprisingly competitive.
- **LongLoRA**. Applies LoRA specifically to extend context length — attention is the bottleneck, and shifted-sparse attention + LoRA lets you extend a 4k model to 100k cheaply.
- **QA-LoRA** / **LoftQ**. Tricks for quantization-aware LoRA that reduce the NF4 → fp16 merge degradation.

If you're starting a project today in 2026, the reasonable defaults are: **DoRA with LoRA+ LR ratio, r=16 or r=32, applied to all linear layers, on an NF4-quantized base**. Everything else is a tuning knob you probably don't need to touch.

## Checkpoint quiz

Test yourself before moving on to Module 18.

1. Why is $B$ initialized to **zero** (and not $A$)?
2. A LoRA-ified layer has $d=4096$ and $r=32$. What fraction of the full matrix's parameter count is trainable?
3. What property of the fine-tuning *update* (not the weights themselves) makes LoRA work?
4. Give two reasons LoRA training is faster per step than full fine-tuning, even ignoring memory.
5. After training, you want zero inference overhead. What operation do you apply to the LoRA module and what does it produce?
6. In QLoRA, the frozen base is in **NF4** but the adapters stay in **bf16**. Why not quantize the adapters too?
7. What does the LoRA $\alpha$ hyperparameter actually control, and why is it divided by $r$?
8. You train LoRA on task A, merge, then train LoRA again on task B on top of the merged model. What have you actually accomplished, and when might this be a bad idea?

<details><summary>Answers</summary>

1. So that at step 0, $BA = 0 \cdot A = 0$, meaning the LoRA path contributes nothing and the model is *exactly* the pretrained base. If both were random, the first forward pass would be degraded before training even started.
2. $\frac{2dr}{d^2} = \frac{2r}{d} = \frac{64}{4096} \approx 1.56\%$. Across a whole model (only wrapping some layers), it ends up being 0.1–1%.
3. It's **low rank**. Empirically the update $\Delta W$ from fine-tuning is well-approximated by a small number of singular vectors — even though $W$ itself is full rank.
4. (a) Optimizer step is tiny: AdamW on a few hundred K params vs hundreds of M. (b) No gradients flow into the frozen base, so `backward()` skips most of the parameter tensors. Optional (c): the frozen base's grad buffers aren't allocated at all.
5. `W_merged = W_0 + (alpha/r) * B @ A`. You replace the LoRA module with a plain linear layer holding `W_merged`. Inference is bit-identical to a regular dense layer.
6. The adapters are where learning happens — they need precision to accumulate small gradient updates. 4-bit adapters would round to zero constantly. Also, they're tiny: you save essentially nothing by quantizing them.
7. $\alpha$ scales the LoRA contribution: $\Delta W = (\alpha/r) BA$. Dividing by $r$ keeps the effective update magnitude roughly constant as you change $r$, so $\alpha$ behaves like a rank-independent learning-rate multiplier on the adapter branch.
8. You've done sequential fine-tuning. It works, but you've lost the ability to independently swap task A vs task B — the merged base now contains task A. For multi-task deployment you want to keep each LoRA separate and let an inference server swap them per request.

</details>

---

**Next up — Module 18: Next-Token Prediction.** Now that you know how to *train* a model efficiently, we'll go back to the core objective that makes an LLM an LLM in the first place.